# 5. From molecules to a machine-learning model

In notebooks 1–4 you learnt how to read structures, search them, put them in a table and describe them with numbers. Here we use those numbers to **train a model**: a program that learns a relationship between molecular features and a known outcome, then predicts that outcome for other molecules.

We will work through one question: can a model use molecular structure to predict the blood–brain barrier penetration label recorded in a dataset? The aim is to understand the complete workflow, from SMILES to an honest evaluation of predictions.

By the end you should be able to:

- distinguish model inputs (**features**) from the outcome (**target**)
- clean missing, unreadable and duplicate molecular records
- keep separate training and test sets
- train a classifier using descriptors and then Morgan fingerprints
- compare it with a simple baseline and explain its mistakes

Work from top to bottom. Cells marked **Exercise** are for you to try. A hidden solution follows each one—click "Solution" to show it, but have a go first! The main lesson runs even if you leave an exercise blank. We will briefly recall material from notebooks 1, 3 and 4 as we use it.

## Setup

**Using Google Colab?** Run the cell below first: click it, then press **Shift + Enter** (or click the ▶ button on its left). It installs RDKit and downloads the data this notebook needs, and takes about 30 seconds. On your own computer it does nothing.

In [ ]:
# Setup for Google Colab (does nothing on your own computer)
import os, sys, urllib.request

if 'google.colab' in sys.modules:
    %pip install -q rdkit
    DATA_URL = 'https://raw.githubusercontent.com/aichemy-hub/rdkit-tutorials/master/data/'
    os.makedirs('../data', exist_ok=True)
    for name in ['bbbp.csv']:
        if not os.path.exists('../data/' + name):
            urllib.request.urlretrieve(DATA_URL + name, '../data/' + name)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay

## 1. What are we trying to predict?

The **blood–brain barrier (BBB)** controls the movement of substances from the blood into the brain. The BBBP dataset contains molecules with a reported binary penetration label. We will predict that label, rather than calculate a property such as molecular weight.

This is **supervised learning**: during training, we provide molecular features and the known label for each molecule. It is a **classification** task because the outcome is a category:

| Column | Meaning |
| --- | --- |
| `name` | A label for the molecule |
| `smiles` | The structure we will read with RDKit |
| `p_np` | Our target: `1` = BBB-penetrating, `0` = non-penetrating |
| `num` | An original row identifier |

The labels are those supplied by this historical dataset. They simplify a complex biological process and may contain uncertainty. Our model will learn patterns in these records; it will not establish whether a new drug is safe or effective.

The local file is a copy of the [BBBP dataset used by MoleculeNet and DeepChem](https://deepchem.readthedocs.io/en/latest/api_reference/moleculenet.html#bbbp-datasets). You can run this lesson without an internet connection.

In [ ]:
raw = pd.read_csv('../data/bbbp.csv')
print('Rows and columns:', raw.shape)
raw.head()

Before calculating anything, look at the label counts. A dataset with many more `1`s than `0`s is **imbalanced**. A model that always predicts `1` might appear successful if we look only at the percentage of correct answers.

In [ ]:
raw['p_np'].value_counts()

### Exercise 1

How many molecules have label `0`? Use `value_counts()` and select `[0]`, as you did for a group count in notebook 4.

In [ ]:
# Exercise 1: your code here

<details>
<summary>Solution</summary>

```python
raw['p_np'].value_counts()[0]
```

The supplied file has 483 label-0 records and 1,567 label-1 records. These are the counts before cleaning.
</details>

## 2. Prepare the molecular records

We need both a readable structure and a known outcome for every training example. A missing name is inconvenient for display, but a missing target cannot tell the model what to learn.

Unlike the ChEMBL examples in notebooks 3 and 4, we will keep both outcome classes and avoid filtering on approval phase or the rule of five. Our question concerns the molecules in this dataset, including molecules that are not approved drugs.

First check for missing entries. Then keep rows with a SMILES and a target of either `0` or `1`. We do not fill in unknown target labels by guessing.

In [ ]:
raw[['smiles', 'p_np']].isna().sum()

In [ ]:
df = raw.dropna(subset=['smiles', 'p_np']).copy()
df = df[df['p_np'].isin([0, 1])].copy()
print('Rows with a structure string and a valid label:', len(df))

### Read the SMILES

As in notebook 1, `Chem.MolFromSmiles()` returns a molecule or `None` when parsing fails. A string can be present in the file and still be unreadable.

The short `with rdBase.BlockLogs():` block temporarily hides RDKit's repeated parsing messages. We still count and inspect every failed record; logging resumes when the block ends.

In [ ]:
with rdBase.BlockLogs():
    df['ROMol'] = df['smiles'].map(Chem.MolFromSmiles)

invalid = df[df['ROMol'].isna()]
print('Unreadable SMILES:', len(invalid))
invalid[['name', 'smiles']].head()

In [ ]:
df = df[df['ROMol'].notna()].copy()
print('Rows RDKit can read:', len(df))

### Identify repeated structures

Notebook 1 showed that different SMILES can describe the same molecule. Checking only for repeated text could miss these duplicates. We therefore ask RDKit to write each molecule as a **canonical SMILES** and use that as a key.

If we let the same molecular record appear in both training and test sets, the test could reward remembering that molecule. Deduplication happens **before** we split the data.

In [ ]:
df['canonical_smiles'] = df['ROMol'].map(Chem.MolToSmiles)
print('Extra rows with a repeated structure:', df['canonical_smiles'].duplicated().sum())

Some repeated structures have **conflicting labels**: the same canonical structure is recorded as both `0` and `1`. We cannot resolve that disagreement from these four original columns.

For this lesson, remove every record belonging to a conflicting structure. For repeated structures whose labels agree, keep one record. This is a transparent teaching choice; a research study would investigate the underlying measurements.

`groupby()` collects rows with the same canonical SMILES, and `nunique()` counts their distinct labels. A count greater than one identifies a conflict.

In [ ]:
label_counts = df.groupby('canonical_smiles')['p_np'].nunique()
conflicting_smiles = label_counts[label_counts > 1].index
print('Structures with conflicting labels:', len(conflicting_smiles))

In [ ]:
df = df[~df['canonical_smiles'].isin(conflicting_smiles)].copy()
df = df.drop_duplicates(subset='canonical_smiles').reset_index(drop=True)
print('Molecules after cleaning:', len(df))
df['p_np'].value_counts()

The salt example in notebook 3 showed why molecular preparation matters. Canonical SMILES identifies repeated structures **as RDKit represents them here**. It does not automatically make a free base, a salt and a different tautomer equivalent. A dot in SMILES can separate fragments, such as a drug and its counterion.

We keep the supplied fragments and protonation states in this introductory model. That means some related forms may remain as separate records. More complete chemical standardisation would require an explicit policy about salts, protonation, tautomers and stereochemistry. Do not delete a fragment simply because it looks inconvenient.

### Exercise 2

Check that the cleaned table has no repeated `canonical_smiles`. Use `.duplicated().sum()`.

In [ ]:
# Exercise 2: your code here

<details>
<summary>Solution</summary>

```python
df['canonical_smiles'].duplicated().sum()
```

The result should be 0. This check does not establish that all related salt or tautomer forms have been merged.
</details>

## 3. Turn structures into features

A **feature** is one input number a model can use. The **target** is the answer we want it to predict. In notebook 4 we used molecular numbers to place points on a plot; here we use them to learn a relationship with `p_np`.

RDKit calculates the features. Scikit-learn trains the model. The model receives a table of numbers, rather than molecule drawings or drug names.

We will start with the four descriptors already calculated in notebook 3: MW, logP, HBD and HBA. Their meanings are familiar, and they offer a simple first representation of each molecule.

In [ ]:
df['MW'] = df['ROMol'].map(Descriptors.MolWt)
df['logP'] = df['ROMol'].map(Descriptors.MolLogP)
df['HBD'] = df['ROMol'].map(Descriptors.NumHDonors)
df['HBA'] = df['ROMol'].map(Descriptors.NumHAcceptors)
df[['name', 'MW', 'logP', 'HBD', 'HBA', 'p_np']].head()

We call the feature table **`X`** and the target column **`y`**, following the usual machine-learning convention. Each row of `X` corresponds to the same row of `y`.

Do not include `p_np` in `X`: that would give the model the answer. We also leave out names and row identifiers. They are labels for our records, rather than the molecular properties we intend to learn from.

In [ ]:
feature_columns = ['MW', 'logP', 'HBD', 'HBA']
X = df[feature_columns]
y = df['p_np']
print('Feature table shape:', X.shape)
print('Target shape:', y.shape)

### Exercise 3

Display the first five rows of `X`. How many input numbers does the model get for each molecule?

In [ ]:
# Exercise 3: your code here

<details>
<summary>Solution</summary>

```python
X.head()
```

Four: molecular weight, calculated logP, hydrogen-bond donors and hydrogen-bond acceptors. The target is stored separately in `y`.
</details>

## 4. Keep a test set aside

Training a model and testing it on the same molecules would not tell us whether it can generalise. We need a **training set** for learning and a separate **test set** for evaluation.

We will set aside 20% of the cleaned molecules. `stratify=y` keeps approximately the same fraction of each label in both sets, and `random_state=42` makes the split repeatable. The number 42 has no chemical meaning.

We split the molecular table so that the descriptor and fingerprint models can later use **exactly the same test molecules**.

In [ ]:
train, test = train_test_split(
    df, test_size=0.2, stratify=y, random_state=42
)
print('Training molecules:', len(train))
print('Test molecules:', len(test))

In [ ]:
X_train = train[feature_columns]
X_test = test[feature_columns]
y_train = train['p_np']
y_test = test['p_np']

This is a **random molecule split**. Exact duplicate canonical structures have been removed, but closely related molecules can still occur on both sides. The score will describe this split; it may be optimistic for predicting entirely new chemical families. We will return to this limitation at the end.

Keep training and test rows in the same order as their corresponding labels. A model cannot recognise that the labels have been accidentally shuffled relative to the feature rows.

### Exercise 4

Write a short answer as a Python comment: why should the model learn from `X_train` and `y_train`, rather than from `X_test` and `y_test`?

In [ ]:
# Exercise 4: write your answer here as a comment

<details>
<summary>Solution</summary>

The test set is our check on molecules the model did not learn from. Training on it would remove that independent check.
</details>

## 5. Start with a simple baseline

Before using chemistry, ask what happens if we always choose the most common training label. This is a **baseline**: a simple reference that our molecular models should improve on.

`DummyClassifier(strategy='most_frequent')` learns which label is most common and predicts it for every molecule. It ignores the molecular feature values. We use `.fit()` to learn from the training set and `.predict()` to make test predictions; the same two methods are used for our real models.

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
print('Baseline accuracy:', round(accuracy_score(y_test, baseline_predictions), 3))

The baseline can be correct on roughly three quarters of this dataset simply by predicting label `1`. That is why a high-looking accuracy needs context. A useful model must also recognise some label-`0` molecules.

We will compare all models using **accuracy** (the fraction of correct answers) and **balanced accuracy** (the average recall for the two classes). Recall for a class is the fraction of its true examples that the model recognises. An always-one classifier has recall 1 for class 1 and recall 0 for class 0, giving balanced accuracy 0.5.

## 6. Train a model on descriptors

A **decision tree** learns questions about input numbers, such as whether a descriptor is above a threshold, to divide training examples into groups. A **random forest** combines many such trees and uses their votes to choose a class. The trees learn the questions from examples and labels; we do not write the chemical rules ourselves.

We use 100 trees (`n_estimators=100`) and limit their depth to 10 (`max_depth=10`) so they cannot keep splitting without limit. These are **model settings**, also called hyperparameters. `random_state` makes training repeatable.

Random forests do not require us to standardise these feature scales. For models that do need a fitted scaler or imputer, fit that preprocessing on the training data and apply it to the test data afterwards.

In [ ]:
descriptor_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42
)
descriptor_model.fit(X_train, y_train)

The forest has now learnt from the descriptor rows and their training labels. It has not seen the test labels during training. `.predict()` returns one predicted label for each test row, in that row order.

In [ ]:
descriptor_predictions = descriptor_model.predict(X_test)
descriptor_predictions[:5]

### Exercise 5

Predict the labels for just the first five rows of `X_test`. Use `descriptor_model.predict(X_test.head())`.

In [ ]:
# Exercise 5: your code here

<details>
<summary>Solution</summary>

```python
descriptor_model.predict(X_test.head())
```

These should match the first five entries printed above. No fitting is needed to make predictions with a trained model.
</details>

## 7. Try Morgan fingerprints as features

Notebook 4 introduced Morgan fingerprints and compared them with Tanimoto similarity. Here, rather than calculate a similarity score, we give the fingerprint bits to a model that learns from the recorded labels. We will reuse them to represent the local patterns of atoms and bonds, giving the model more structural information than our four descriptors.

We keep radius 2 and use **1,024 bits**. This is longer than the 256-bit teaching example in notebook 4, giving more room to store patterns, though different patterns can still map to the same bit. These settings do not explicitly include tetrahedral chirality.

We will use the **full fingerprints**, without reducing them to two PCA coordinates. PCA helped us draw a map, but a model can use all 1,024 input columns. Train and test molecules must use the same generator.

In [ ]:
fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

train_fingerprints = train['ROMol'].map(fingerprint_generator.GetFingerprintAsNumPy)
test_fingerprints = test['ROMol'].map(fingerprint_generator.GetFingerprintAsNumPy)
X_train_fp = np.array(train_fingerprints.to_list())
X_test_fp = np.array(test_fingerprints.to_list())
print('Training fingerprint matrix:', X_train_fp.shape)
print('Test fingerprint matrix:', X_test_fp.shape)

### Train the fingerprint model

Now train a second forest, using the same settings and the same training molecules as before. Only the feature representation changes. This makes the comparison with our descriptor model easier to interpret.

In [ ]:
fingerprint_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42
)
fingerprint_model.fit(X_train_fp, y_train)
fingerprint_predictions = fingerprint_model.predict(X_test_fp)

## 8. Compare predictions with the known test labels

The two models and the baseline make predictions for the same test molecules. We can compare those predictions with `y_test`, the labels held back from training.

Accuracy counts all correct predictions equally. Balanced accuracy gives the two classes equal weight, even though label 0 has fewer examples. Inspect both rather than deciding that one model is best from a single number. This is one reproducible split, so small score differences need more investigation.

In [ ]:
scores = pd.DataFrame({
    'Model': ['Most frequent label', 'Four descriptors', 'Morgan fingerprints'],
    'Accuracy': [
        accuracy_score(y_test, baseline_predictions),
        accuracy_score(y_test, descriptor_predictions),
        accuracy_score(y_test, fingerprint_predictions)
    ],
    'Balanced accuracy': [
        balanced_accuracy_score(y_test, baseline_predictions),
        balanced_accuracy_score(y_test, descriptor_predictions),
        balanced_accuracy_score(y_test, fingerprint_predictions)
    ]
})
scores.round(3)

Read both columns of the score table. A model can have higher accuracy but lower balanced accuracy if its success is concentrated in the more common class. More fingerprint columns do not guarantee better predictions for both labels. Which model is preferable depends on the kinds of mistake that matter for the task.

### Where does the fingerprint model make mistakes?

A **confusion matrix** counts correct and incorrect answers separately for each class. Read its **rows as the known label** and its **columns as the model prediction**.

- Upper left: label 0 correctly predicted as 0.
- Upper right: label 0 incorrectly predicted as 1.
- Lower left: label 1 incorrectly predicted as 0.
- Lower right: label 1 correctly predicted as 1.

The two kinds of mistake may have different consequences in a real project. Here the matrix helps us see what a single accuracy number hides.

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, fingerprint_predictions,
    labels=[0, 1], display_labels=['BBB− (0)', 'BBB+ (1)'], cmap='Blues'
)
plt.title('Fingerprint model: held-out test molecules')
plt.show()

We can also put the predictions next to the original molecular records. Inspecting incorrect examples can suggest questions about label quality, unusual structures or the limits of our features. A wrong prediction alone is not a reason to delete a molecule from the dataset.

In [ ]:
test_results = test[['name', 'smiles', 'p_np']].copy()
test_results['predicted'] = fingerprint_predictions
test_results[test_results['p_np'] != test_results['predicted']].head()

### Exercise 6

How many test predictions are wrong? Compare `y_test` with `fingerprint_predictions` using `!=`, then add up the `True` values with `.sum()`.

In [ ]:
# Exercise 6: your code here

<details>
<summary>Solution</summary>

```python
(y_test != fingerprint_predictions).sum()
```

This should equal the sum of the two off-diagonal entries in the confusion matrix.
</details>

### Training performance is not test performance

A model may fit its training examples much more closely than it fits unseen molecules. This is called **overfitting**. Compare the two accuracies below: a high training score by itself does not establish good prediction.

These scores are checks on our model, rather than instructions to keep adjusting settings until this particular test score rises.

In [ ]:
training_predictions = fingerprint_model.predict(X_train_fp)
print('Training accuracy:', round(accuracy_score(y_train, training_predictions), 3))
print('Test accuracy:', round(accuracy_score(y_test, fingerprint_predictions), 3))

## 9. Predict from one SMILES string

The full path is now: **SMILES → RDKit molecule → fingerprint → trained model → prediction**. To practise receiving a single structure, we take one molecule from the held-out test set. It is already known to parse correctly.

A model expects a table even when there is only one molecule. `np.array([one_fingerprint])` creates a table with one row and 1,024 columns. We reuse the same generator and trained model; we do not train again on this example.

In [ ]:
one_smiles = test['smiles'].iloc[0]
one_mol = Chem.MolFromSmiles(one_smiles)
one_fingerprint = fingerprint_generator.GetFingerprintAsNumPy(one_mol)
one_X = np.array([one_fingerprint])

print('Molecule:', test['name'].iloc[0])
print('Predicted label:', fingerprint_model.predict(one_X)[0])
print('Recorded label:', test['p_np'].iloc[0])

### Exercise 7

Write a short answer as a Python comment: why must you use the same fingerprint settings for a new molecule as you used during training?

In [ ]:
# Exercise 7: write your answer here as a comment

<details>
<summary>Solution</summary>

The trained model learnt relationships for a particular set of fingerprint columns. Changing the length, radius or other settings changes those columns, so the old model would no longer receive the representation it learnt from.
</details>

## 10. Optional: check a model on several training splits

If we want to compare model settings, use the **training data** for those decisions. **Cross-validation** repeatedly holds back part of the training set, trains on the rest, and checks predictions on the held-back part. It gives us several validation scores instead of relying on one training/validation split.

The three-fold example below creates three such checks. Scikit-learn fits fresh copies of the classifier for them, so the already-trained `fingerprint_model` is not replaced. We use balanced accuracy because both classes matter.

If you try different values of `max_depth` or different features, compare their training cross-validation scores. Keep the test set for a final assessment of the chosen workflow; repeated choices based on test scores make it less independent.

In [ ]:
validation_scores = cross_val_score(
    fingerprint_model, X_train_fp, y_train,
    cv=3, scoring='balanced_accuracy'
)
print('Training cross-validation scores:', validation_scores.round(3))
print('Mean validation score:', round(validation_scores.mean(), 3))

## Summary: what we learnt and what remains

RDKit supplied features derived from structure. Scikit-learn learnt a relationship between those features and the recorded target labels. We checked the resulting predictions against a baseline and held-out molecules. The full process is more informative than reporting a training score alone.

Several choices still limit what our numbers tell us:

- **The split:** related structures can appear in both sides of a random split. A scaffold split groups molecules by their core ring-and-linker framework before separating training and test data. This is a useful next step when asking about new chemical families; [MoleculeNet recommends scaffold splitting for BBBP](https://deepchem.readthedocs.io/en/latest/api_reference/moleculenet.html#bbbp-datasets).
- **Molecular preparation:** canonical deduplication here does not merge related salts or tautomers. Our fingerprints also do not explicitly encode stereochemistry or 3D structure.
- **Labels and scope:** the model learns the records available here. Conflicting, uncertain or unrepresentative labels limit its predictions.
- **Feature choices:** descriptors and fingerprints offer different information. Combining them or changing their settings requires another properly validated comparison.

For a first model, knowing what was learnt, what was held back and what the input numbers represent matters more than obtaining a particular score.

Optional references: [RDKit fingerprints](https://www.rdkit.org/docs/GettingStartedInPython.html#morgan-fingerprints-circular-fingerprints), [scikit-learn random forests](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html), and [scikit-learn model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html).

This is the last notebook in the course. Well done for working through it!

*Written for this course by Pablo Graf and Dylan John, adapted from a feature-engineering and BBBP workshop exercise. Licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).*